# Multimodal — exploration et agrégation des résultats

Premières versions métriques + CodeBERT (IsolationForest, sous-échantillonnage, BorderlineSMOTE). La dernière cellule agrège les résultats par projet (`all_projects_results.pkl`) et produit les figures `comparison_<projet>.png` reprises dans `results/figures/`.

**Statut :** Exploratoire + agrégation

*Mémoire de maîtrise — Mireille Kafando, UQTR · 2025*

> Notebook de recherche conçu pour Google Colab. Les chemins `/content/drive/MyDrive/bug/...` sont à adapter (voir `data/README.md`). Les sorties ont été retirées.


# Mounting drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

# Sous échantillonnage avec suppression avec des outliers avec IsolationForest et utilisation de lazylayer dans le modèle

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gmean
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 100
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 40
    N_HEADS = 4
    NUM_LAYERS = 5
    DROPOUT = 0.3
    NUM_CLASSES = 2

class BugDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

class TransModel(nn.Module):
    def __init__(self):
        super(TransModel, self).__init__()

        self.embedding = Config.EMBEDDING_DIM

        # Utilisation de LazyLinear au lieu de Linear
        self.enc_embedding = nn.Sequential(
            nn.LazyLinear(self.embedding, bias=False),
            nn.LazyLinear(self.embedding, bias=False),
            nn.LayerNorm(self.embedding)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embedding,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        self.encoder = nn.Sequential(
            self.enc_embedding,
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=Config.NUM_LAYERS,
                norm=nn.LayerNorm(self.embedding)
            )
        )

        # Utilisation de LazyLinear au lieu de Linear dans le classifieur
        self.classifier = nn.Sequential(
            nn.LazyLinear(4 * self.embedding),
            nn.ReLU(),
            nn.LazyLinear(Config.NUM_CLASSES),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        encoded = self.encoder(x)
        encoded = encoded.squeeze(1)
        return self.classifier(encoded)

def analyze_outlier_detection(X, y, contaminations=[0.05, 0.1, 0.15, 0.2]):
    """
    Analyse l'impact des différents paramètres sur la détection des outliers
    avec IsolationForest et visualise les résultats.
    """
    # Créer un DataFrame avec les données
    df = pd.DataFrame(X)
    df['target'] = y

    # Séparer la classe majoritaire
    majority_class = df[df['target'] == df['target'].value_counts().index[0]]
    minority_class = df[df['target'] == df['target'].value_counts().index[1]]

    # Supprimer la colonne target pour l'analyse
    X_majority = majority_class.drop('target', axis=1)

    # Réduire la dimensionnalité pour la visualisation
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_majority)

    # Créer une grille de sous-plots pour différentes valeurs de contamination
    fig, axes = plt.subplots(1, len(contaminations), figsize=(15, 5))
    fig.suptitle('Analyse de la détection des outliers avec IsolationForest', fontsize=16)

    results = []

    for i, cont in enumerate(contaminations):
        try:
            # Remplacer EllipticEnvelope par IsolationForest
            from sklearn.ensemble import IsolationForest

            detector = IsolationForest(
                contamination=cont,
                n_estimators=100,  # Nombre d'arbres dans la forêt
                max_samples='auto',  # Taille des sous-échantillons
                random_state=42,
                n_jobs=-1  # Utiliser tous les processeurs disponibles
            )

            # Détecter les outliers
            labels = detector.fit_predict(X_majority)
            # Convertir les labels -1 (outliers) et 1 (inliers) en 0 (outliers) et 1 (inliers)
            labels_binary = (labels == 1).astype(int)
            n_outliers = (labels == -1).sum()

            # Stocker les résultats
            results.append({
                'contamination': cont,
                'n_outliers': n_outliers,
                'percent_outliers': (n_outliers / len(X_majority)) * 100
            })

            # Visualiser les résultats
            scatter = axes[i].scatter(X_pca[:, 0], X_pca[:, 1],
                                      c=labels, cmap='viridis')
            axes[i].set_title(f'Contamination={cont}\n'
                             f'Outliers: {n_outliers} ({(n_outliers/len(X_majority))*100:.1f}%)')
            axes[i].grid(True)
        except Exception as e:
            print(f"Erreur avec contamination={cont}: {str(e)}")
            axes[i].text(0.5, 0.5, f"Erreur: {str(e)}",
                        horizontalalignment='center',
                        verticalalignment='center',
                        transform=axes[i].transAxes)
            continue

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)  # Ajuster pour le titre
    plt.show()

    # Afficher un tableau récapitulatif
    results_df = pd.DataFrame(results)
    print("\nRécapitulatif des résultats:")
    print(results_df)

    return results_df

def optimize_outlier_parameters(X, y, initial_majority_ratio=None):
    """
    Trouve les meilleurs paramètres pour la détection des outliers avec IsolationForest
    """
    if initial_majority_ratio is None:
        class_counts = pd.Series(y).value_counts()
        initial_majority_ratio = class_counts[0] / len(y)

    # Paramètres à tester pour IsolationForest
    contaminations = np.linspace(0.05, 0.3, 6)
    n_estimators_options = [50, 100, 200]  # Nombre d'arbres à tester

    best_params = None
    best_score = float('inf')
    target_ratio = 0.5

    results = []

    for cont in contaminations:
        for n_est in n_estimators_options:
            try:
                detector = IsolationForest(
                    contamination=cont,
                    n_estimators=n_est,
                    max_samples='auto',
                    random_state=42,
                    n_jobs=-1
                )

                labels = detector.fit_predict(X)
                # Les outliers sont marqués par -1, convertir en indices binaires
                clean_indices = labels == 1

                new_y = y[clean_indices]
                if len(new_y) == 0:
                    continue

                new_ratio = pd.Series(new_y).value_counts().iloc[0] / len(new_y)
                score = abs(new_ratio - target_ratio)

                results.append({
                    'contamination': cont,
                    'n_estimators': n_est,
                    'new_ratio': new_ratio,
                    'score': score,
                    'remaining_samples': len(new_y)
                })

                if score < best_score:
                    best_score = score
                    best_params = {
                        'contamination': cont,
                        'n_estimators': n_est
                    }
            except Exception as e:
                print(f"Erreur avec cont={cont}, n_est={n_est}: {str(e)}")
                continue

    if not best_params:
        return {'contamination': 0.1, 'n_estimators': 100}, pd.DataFrame()

    results_df = pd.DataFrame(results)

    if not results_df.empty:
        plt.figure(figsize=(8, 4))
        pivot = results_df.pivot(index='contamination', columns='n_estimators', values='score')
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
        plt.title('Scores des combinaisons de paramètres\n(Plus bas = meilleur)')
        plt.show()

    print("\nMeilleurs paramètres trouvés:")
    print(f"Contamination: {best_params['contamination']:.3f}")
    print(f"Nombre d'estimateurs: {best_params['n_estimators']}")

    return best_params, results_df

def remove_outliers_undersampling(X, y, contamination=None, n_estimators=None):
    """
    Version améliorée de la fonction de sous-échantillonnage utilisant IsolationForest
    """
    from sklearn.ensemble import IsolationForest

    if contamination is None or n_estimators is None:
        # Optimiser les paramètres si non fournis
        best_params, _ = optimize_outlier_parameters(X, y)
        contamination = best_params['contamination']
        n_estimators = best_params['n_estimators']

    # Créer le DataFrame
    df = pd.DataFrame(X)
    df['target'] = y

    # Séparer les classes
    majority_class = df[df['target'] == df['target'].value_counts().index[0]]
    minority_class = df[df['target'] == df['target'].value_counts().index[1]]

    # Appliquer IsolationForest avec les paramètres optimisés
    detector = IsolationForest(
        contamination=contamination,
        n_estimators=n_estimators,
        max_samples='auto',
        random_state=42,
        n_jobs=-1
    )

    # Détecter les outliers dans la classe majoritaire
    X_majority = majority_class.drop('target', axis=1)
    labels = detector.fit_predict(X_majority)

    # Garder les échantillons non-outliers (marqués par 1)
    clean_majority = majority_class[labels == 1]

    # Équilibrer si nécessaire
    if len(clean_majority) > len(minority_class):
        clean_majority = clean_majority.sample(n=len(minority_class), random_state=42)

    # Combiner les classes
    balanced_df = pd.concat([clean_majority, minority_class])
    balanced_df = balanced_df.sample(frac=1, random_state=42)

    return balanced_df.drop('target', axis=1).values, balanced_df['target'].values

def load_and_preprocess_data(train_files, val_file):
    # Chargement des données d'entraînement
    train_dfs = []
    for file in train_files:
        df = pd.read_csv(file, usecols=lambda x: x != 'name')
        train_dfs.append(df)
    df_train = pd.concat(train_dfs, ignore_index=True)

    # Chargement des données de validation
    df_val = pd.read_csv(val_file, usecols=lambda x: x != 'name')

    # Conversion en labels binaires
    df_train['bug'] = (df_train['bug'] > 0).astype(int)
    df_val['bug'] = (df_val['bug'] > 0).astype(int)

    # Standardisation et prétraitement
    feature_cols = [col for col in df_train.columns if col != 'bug']
    means_tr = df_train[feature_cols].mean(axis=0)
    stds_tr = df_train[feature_cols].std(axis=0)
    means_val = df_val[feature_cols].mean(axis=0)
    stds_val = df_val[feature_cols].std(axis=0)

    # Suppression des colonnes avec faible écart-type
    cols_to_drop_training = stds_tr[abs(stds_tr) <= 1e-5].index.values
    cols_to_drop_val = stds_val[abs(stds_val) <= 1e-5].index.values
    df_train = df_train.drop(cols_to_drop_training, axis=1)
    df_val = df_val.drop(cols_to_drop_val, axis=1)

    # Mise à jour des colonnes de features
    feature_cols_train = [col for col in df_train.columns if col != 'bug']
    feature_cols_val = [col for col in df_val.columns if col != 'bug']

    # Standardisation des features
    df_train[feature_cols_train] = (df_train[feature_cols_train] - means_tr[feature_cols_train]) / stds_tr[feature_cols_train]
    df_val[feature_cols_val] = (df_val[feature_cols_val] - means_val[feature_cols_val]) / stds_val[feature_cols_val]

    print("\nAnalyse des outliers et optimisation des paramètres avec IsolationForest...")

    # Analyse des paramètres de détection des outliers
    results_df = analyze_outlier_detection(
        df_train.iloc[:, :-1].values,
        df_train.iloc[:, -1].values
    )

    # Optimisation des paramètres
    best_params, _ = optimize_outlier_parameters(
        df_train.iloc[:, :-1].values,
        df_train.iloc[:, -1].values
    )

    print("\nApplication du sous-échantillonnage basé sur les outliers avec IsolationForest...")
    # Application du sous-échantillonnage avec les paramètres optimisés
    X_train_resampled, y_train_resampled = remove_outliers_undersampling(
        df_train.iloc[:, :-1].values,
        df_train.iloc[:, -1].values,
        contamination=best_params['contamination'],
        n_estimators=best_params['n_estimators']
    )

    # Préparation des données finales
    X_val_final = df_val[feature_cols_val].values
    y_val_final = df_val['bug'].values

    print(f"\nStatistiques finales:")
    print(f"Training samples before undersampling: {len(df_train)}")
    print(f"Training samples after outlier-based undersampling: {len(X_train_resampled)}")
    print(f"Features in training set: {X_train_resampled.shape[1]}")
    print(f"Features in validation set: {X_val_final.shape[1]}")

    return X_train_resampled, y_train_resampled, X_val_final, y_val_final

def calculate_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    g_mean = gmean([sensitivity, specificity])

    return cm, sensitivity, specificity, g_mean

def plot_confusion_matrix(cm, title='Confusion Matrix'):
    plt.figure(figsize=(7, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

def train_and_validate():
    train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-3.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.0.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.1.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.2.csv',
    '/content/drive/MyDrive/bug/bug-data/jedit/jedit-4.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.3.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/ant/ant-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/ivy/ivy-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/log4j/log4j-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.2.csv',
    '/content/drive/MyDrive/bug/bug-data/lucene/lucene-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.0.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/poi/poi-3.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/synapse/synapse-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.4.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.5.csv',
    '/content/drive/MyDrive/bug/bug-data/velocity/velocity-1.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.4.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.5.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.6.csv',
    '/content/drive/MyDrive/bug/bug-data/xalan/xalan-2.7.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.3.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.4.4.csv',



    ]

    val_file_path = '/content/drive/MyDrive/bug/bug-data/ant/ant-1.7.csv'
    X_train, y_train, X_val, y_val = load_and_preprocess_data(train_file_paths, val_file_path)

    train_dataset = BugDataset(X_train, y_train)
    val_dataset = BugDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    # Remarquez qu'on n'a plus besoin de spécifier la dimension d'entrée
    model = TransModel().to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for features, labels in train_loader:
            features = features.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss /= len(train_loader)
        train_acc = 100 * train_correct / train_total

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(Config.DEVICE)
                labels = labels.to(Config.DEVICE)

                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        val_acc = 100 * val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print("-" * 50)

    cm, sensitivity, specificity, g_mean = calculate_metrics(all_labels, all_preds)

    plt.figure(figsize=(7, 3))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()

    plt.tight_layout()
    plt.show()

    plot_confusion_matrix(cm)

    print("\nFinal Metrics:")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"G-mean: {g_mean:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

if __name__ == "__main__":
    train_and_validate()

# Suppression avec des outliers et utilisation de smote

In [ ]:
!pip install imbalanced-learn

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gmean
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE

class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 100
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 40
    N_HEADS = 4
    NUM_LAYERS = 5
    DROPOUT = 0.3
    NUM_CLASSES = 2

class BugDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

class TransModel(nn.Module):
    def __init__(self):
        super(TransModel, self).__init__()

        self.embedding = Config.EMBEDDING_DIM

        # Utilisation de LazyLinear au lieu de Linear
        self.enc_embedding = nn.Sequential(
            nn.LazyLinear(self.embedding, bias=False),
            nn.LazyLinear(self.embedding, bias=False),
            nn.LayerNorm(self.embedding)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embedding,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        self.encoder = nn.Sequential(
            self.enc_embedding,
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=Config.NUM_LAYERS,
                norm=nn.LayerNorm(self.embedding)
            )
        )

        # Utilisation de LazyLinear au lieu de Linear dans le classifieur
        self.classifier = nn.Sequential(
            nn.LazyLinear(4 * self.embedding),
            nn.ReLU(),
            nn.LazyLinear(Config.NUM_CLASSES),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        encoded = self.encoder(x)
        encoded = encoded.squeeze(1)
        return self.classifier(encoded)

def remove_outliers_with_isolation_forest(X, y, contamination=0.1, n_estimators=100):
    """
    Supprimer les outliers avec IsolationForest

    Args:
        X (numpy.ndarray): Features
        y (numpy.ndarray): Labels
        contamination (float): Proportion attendue d'outliers
        n_estimators (int): Nombre d'arbres dans la forêt

    Returns:
        tuple: Features et labels nettoyés
    """
    # Créer le détecteur d'outliers
    detector = IsolationForest(
        contamination=contamination,
        n_estimators=n_estimators,
        max_samples='auto',
        random_state=42,
        n_jobs=-1
    )

    # Prédire les outliers
    outlier_labels = detector.fit_predict(X)

    # Garder uniquement les inliers (labels == 1)
    mask = outlier_labels == 1
    X_cleaned = X[mask]
    y_cleaned = y[mask]

    # Visualisation des outliers (projection PCA)
    plt.figure(figsize=(10, 5))

    # Projection PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    plt.subplot(1, 2, 1)
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=outlier_labels, cmap='viridis')
    plt.title('Outliers Detection\n(Blue: Inliers, Yellow: Outliers)')
    plt.xlabel('First Principal Component')
    plt.ylabel('Second Principal Component')

    # Nombre et pourcentage d'outliers
    n_outliers = len(X) - len(X_cleaned)
    outlier_percentage = (n_outliers / len(X)) * 100

    plt.subplot(1, 2, 2)
    plt.bar(['Inliers', 'Outliers'], [len(X_cleaned), n_outliers])
    plt.title(f'Outliers Distribution\n({outlier_percentage:.2f}% Outliers)')

    plt.tight_layout()
    plt.show()

    print(f"\nOutliers removed: {n_outliers} ({outlier_percentage:.2f}%)")

    return X_cleaned, y_cleaned

def load_and_preprocess_data(train_files, val_file):
    # Chargement des données d'entraînement
    train_dfs = []
    for file in train_files:
        df = pd.read_csv(file, usecols=lambda x: x != 'name')
        train_dfs.append(df)
    df_train = pd.concat(train_dfs, ignore_index=True)

    # Chargement des données de validation
    df_val = pd.read_csv(val_file, usecols=lambda x: x != 'name')

    # Conversion en labels binaires
    df_train['bug'] = (df_train['bug'] > 0).astype(int)
    df_val['bug'] = (df_val['bug'] > 0).astype(int)

    # Standardisation et prétraitement
    feature_cols = [col for col in df_train.columns if col != 'bug']
    means_tr = df_train[feature_cols].mean(axis=0)
    stds_tr = df_train[feature_cols].std(axis=0)
    means_val = df_val[feature_cols].mean(axis=0)
    stds_val = df_val[feature_cols].std(axis=0)

    # Suppression des colonnes avec faible écart-type
    cols_to_drop_training = stds_tr[abs(stds_tr) <= 1e-5].index.values
    cols_to_drop_val = stds_val[abs(stds_val) <= 1e-5].index.values
    df_train = df_train.drop(cols_to_drop_training, axis=1)
    df_val = df_val.drop(cols_to_drop_val, axis=1)

    # Mise à jour des colonnes de features
    feature_cols_train = [col for col in df_train.columns if col != 'bug']
    feature_cols_val = [col for col in df_val.columns if col != 'bug']

    # Standardisation des features
    df_train[feature_cols_train] = (df_train[feature_cols_train] - means_tr[feature_cols_train]) / stds_tr[feature_cols_train]
    df_val[feature_cols_val] = (df_val[feature_cols_val] - means_val[feature_cols_val]) / stds_val[feature_cols_val]

    # Préparation des données
    X_train = df_train[feature_cols_train].values
    y_train = df_train['bug'].values
    X_val = df_val[feature_cols_val].values
    y_val = df_val['bug'].values

    # Supprimer les outliers
    X_train_cleaned, y_train_cleaned = remove_outliers_with_isolation_forest(X_train, y_train)

    # Appliquer SMOTE pour l'équilibrage des classes
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_cleaned, y_train_cleaned)

    print(f"\nStatistiques de prétraitement:")
    print(f"Training samples before cleaning: {len(X_train)}")
    print(f"Training samples after cleaning: {len(X_train_cleaned)}")
    print(f"Training samples after SMOTE: {len(X_train_balanced)}")
    print(f"Class distribution after SMOTE: {np.unique(y_train_balanced, return_counts=True)[1]}")

    return X_train_balanced, y_train_balanced, X_val, y_val

def calculate_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    g_mean = gmean([sensitivity, specificity])

    return cm, sensitivity, specificity, g_mean

def plot_confusion_matrix(cm, title='Confusion Matrix'):
    plt.figure(figsize=(7, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

def train_and_validate():
    train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.1.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.3.csv',





    ]

    val_file_path = '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.4.4.csv'
    X_train, y_train, X_val, y_val = load_and_preprocess_data(train_file_paths, val_file_path)

    train_dataset = BugDataset(X_train, y_train)
    val_dataset = BugDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    model = TransModel().to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for features, labels in train_loader:
            features = features.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss /= len(train_loader)
        train_acc = 100 * train_correct / train_total

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(Config.DEVICE)
                labels = labels.to(Config.DEVICE)

                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        val_acc = 100 * val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print("-" * 50)

    cm, sensitivity, specificity, g_mean = calculate_metrics(all_labels, all_preds)

    plt.figure(figsize=(7, 3))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()

    plt.tight_layout()
    plt.show()

    plot_confusion_matrix(cm)

    print("\nFinal Metrics:")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"G-mean: {g_mean:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

if __name__ == "__main__":
    train_and_validate()

# Suppression avec des outliers sans équilibrage des classes

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gmean
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 100
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 40
    N_HEADS = 4
    NUM_LAYERS = 5
    DROPOUT = 0.3
    NUM_CLASSES = 2

class BugDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

class TransModel(nn.Module):
    def __init__(self):
        super(TransModel, self).__init__()

        self.embedding = Config.EMBEDDING_DIM

        # Utilisation de LazyLinear au lieu de Linear
        self.enc_embedding = nn.Sequential(
            nn.LazyLinear(self.embedding, bias=False),
            nn.LazyLinear(self.embedding, bias=False),
            nn.LayerNorm(self.embedding)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embedding,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        self.encoder = nn.Sequential(
            self.enc_embedding,
            nn.TransformerEncoder(
                encoder_layer,
                num_layers=Config.NUM_LAYERS,
                norm=nn.LayerNorm(self.embedding)
            )
        )

        # Utilisation de LazyLinear au lieu de Linear dans le classifieur
        self.classifier = nn.Sequential(
            nn.LazyLinear(4 * self.embedding),
            nn.ReLU(),
            nn.LazyLinear(Config.NUM_CLASSES),
            nn.Softmax(dim=-1)
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        encoded = self.encoder(x)
        encoded = encoded.squeeze(1)
        return self.classifier(encoded)

def remove_outliers(X, y, contamination=0.1, n_estimators=100):
    """
    Supprimer les outliers avec IsolationForest sans sous-échantillonnage
    """
    detector = IsolationForest(
        contamination=contamination,
        n_estimators=n_estimators,
        max_samples='auto',
        random_state=42,
        n_jobs=-1
    )

    # Détecter les outliers dans tout le jeu de données
    labels = detector.fit_predict(X)

    # Garder uniquement les inliers (labels = 1)
    inlier_mask = labels == 1
    X_clean = X[inlier_mask]
    y_clean = y[inlier_mask]

    return X_clean, y_clean

def load_and_preprocess_data(train_files, val_file):
    # Chargement des données d'entraînement
    train_dfs = []
    for file in train_files:
        df = pd.read_csv(file, usecols=lambda x: x != 'name')
        train_dfs.append(df)
    df_train = pd.concat(train_dfs, ignore_index=True)

    # Chargement des données de validation
    df_val = pd.read_csv(val_file, usecols=lambda x: x != 'name')

    # Conversion en labels binaires
    df_train['bug'] = (df_train['bug'] > 0).astype(int)
    df_val['bug'] = (df_val['bug'] > 0).astype(int)

    # Standardisation et prétraitement
    feature_cols = [col for col in df_train.columns if col != 'bug']
    means_tr = df_train[feature_cols].mean(axis=0)
    stds_tr = df_train[feature_cols].std(axis=0)
    means_val = df_val[feature_cols].mean(axis=0)
    stds_val = df_val[feature_cols].std(axis=0)

    # Suppression des colonnes avec faible écart-type
    cols_to_drop_training = stds_tr[abs(stds_tr) <= 1e-5].index.values
    cols_to_drop_val = stds_val[abs(stds_val) <= 1e-5].index.values
    df_train = df_train.drop(cols_to_drop_training, axis=1)
    df_val = df_val.drop(cols_to_drop_val, axis=1)

    # Mise à jour des colonnes de features
    feature_cols_train = [col for col in df_train.columns if col != 'bug']
    feature_cols_val = [col for col in df_val.columns if col != 'bug']

    # Standardisation des features
    df_train[feature_cols_train] = (df_train[feature_cols_train] - means_tr[feature_cols_train]) / stds_tr[feature_cols_train]
    df_val[feature_cols_val] = (df_val[feature_cols_val] - means_val[feature_cols_val]) / stds_val[feature_cols_val]

    # Suppression des outliers de l'ensemble d'entraînement
    X_train_clean, y_train_clean = remove_outliers(
        df_train.iloc[:, :-1].values,
        df_train.iloc[:, -1].values
    )

    # Préparation des données finales
    X_val_final = df_val[feature_cols_val].values
    y_val_final = df_val['bug'].values

    print(f"\nStatistiques finales:")
    print(f"Training samples before outlier removal: {len(df_train)}")
    print(f"Training samples after outlier removal: {len(X_train_clean)}")
    print(f"Features in training set: {X_train_clean.shape[1]}")
    print(f"Features in validation set: {X_val_final.shape[1]}")

    return X_train_clean, y_train_clean, X_val_final, y_val_final

def calculate_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    g_mean = gmean([sensitivity, specificity])

    return cm, sensitivity, specificity, g_mean

def plot_confusion_matrix(cm, title='Confusion Matrix'):
    plt.figure(figsize=(7, 3))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

def train_and_validate():
    train_file_paths = [
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.0.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.2.csv',
    '/content/drive/MyDrive/bug/bug-data/camel/camel-1.4.csv',


    ]

    val_file_path = '/content/drive/MyDrive/bug/bug-data/camel/camel-1.6.csv'
    X_train, y_train, X_val, y_val = load_and_preprocess_data(train_file_paths, val_file_path)

    train_dataset = BugDataset(X_train, y_train)
    val_dataset = BugDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    model = TransModel().to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for features, labels in train_loader:
            features = features.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss /= len(train_loader)
        train_acc = 100 * train_correct / train_total

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(Config.DEVICE)
                labels = labels.to(Config.DEVICE)

                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        val_acc = 100 * val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print("-" * 50)

    cm, sensitivity, specificity, g_mean = calculate_metrics(all_labels, all_preds)

    plt.figure(figsize=(7, 3))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()

    plt.tight_layout()
    plt.show()

    plot_confusion_matrix(cm)

    print("\nFinal Metrics:")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"G-mean: {g_mean:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

if __name__ == "__main__":
    train_and_validate()

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import wandb
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gmean
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 100
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-5
    EMBEDDING_DIM = 40
    N_HEADS = 4
    NUM_LAYERS = 5
    DROPOUT = 0.3
    NUM_CLASSES = 2

    # Wandb configuration
    PROJECT_NAME = "bug-detection-transformer"
    RUN_NAME = "transformer-v1"

# [Rest of the previous code remains the same until train_and_validate function]

def train_and_validate():
    # Initialize wandb
    wandb.init(
        project=Config.PROJECT_NAME,
        name=Config.RUN_NAME,
        config={
            "learning_rate": Config.LEARNING_RATE,
            "epochs": Config.EPOCHS,
            "batch_size": Config.BATCH_SIZE,
            "embedding_dim": Config.EMBEDDING_DIM,
            "n_heads": Config.N_HEADS,
            "num_layers": Config.NUM_LAYERS,
            "dropout": Config.DROPOUT
        }
    )

    train_file_paths = [
        '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.1.csv',
        '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.2.csv',
        '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.3.csv',
    ]

    val_file_path = '/content/drive/MyDrive/bug/bug-data/xerces/xerces-1.4.4.csv'
    X_train, y_train, X_val, y_val = load_and_preprocess_data(train_file_paths, val_file_path)

    train_dataset = BugDataset(X_train, y_train)
    val_dataset = BugDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE)

    model = TransModel().to(Config.DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

    # Watch model with wandb
    wandb.watch(model, log="all", log_freq=10)

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    for epoch in range(Config.EPOCHS):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for features, labels in train_loader:
            features = features.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_loss /= len(train_loader)
        train_acc = 100 * train_correct / train_total

        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(Config.DEVICE)
                labels = labels.to(Config.DEVICE)

                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        val_loss /= len(val_loader)
        val_acc = 100 * val_correct / val_total

        # Log metrics to wandb
        wandb.log({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc
        })

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print("-" * 50)

    # Calculate final metrics
    cm, sensitivity, specificity, g_mean = calculate_metrics(all_labels, all_preds)

    # Log confusion matrix to wandb
    wandb_cm = wandb.Image(
        plt.figure(figsize=(7, 3)),
        caption="Confusion Matrix"
    )

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')

    # Log final metrics to wandb
    wandb.log({
        "confusion_matrix": wandb_cm,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "g_mean": g_mean,
        "classification_report": classification_report(all_labels, all_preds)
    })

    # Close wandb run
    wandb.finish()

    # Optional: local visualization (can be kept or removed)
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Accuracy')
    plt.plot(history['val_acc'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()

    plt.tight_layout()
    plt.show()

    plot_confusion_matrix(cm)

    print("\nFinal Metrics:")
    print(f"Sensitivity: {sensitivity:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"G-mean: {g_mean:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

if __name__ == "__main__":
    # Login to wandb (you'll need to input your API key)
    wandb.login()

    # Run the training
    train_and_validate()

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-7
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

        # Vérifier la cohérence des tailles
        assert len(self.metrics) == len(self.labels), "Mismatch entre metrics et labels"
        assert len(self.code_tokens['input_ids']) == len(self.labels), "Mismatch entre code_tokens et labels"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# STRATÉGIE 1: Modèle hybride (encoder from scratch + decoder pré-entraîné)
class HybridTransBugModel(nn.Module):
    def __init__(self, input_metric_dim=None):
        super().__init__()
        # Projection des métriques vers la dimension d'embedding
        self.metric_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM, bias = False) if input_metric_dim is None else nn.Linear(input_metric_dim, Config.EMBEDDING_DIM, bias = False),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder transformer FROM SCRATCH pour les métriques
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=Config.EMBEDDING_DIM,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            activation='relu'

        )
        self.metrics_encoder = nn.TransformerEncoder(encoder_layer, num_layers=Config.NUM_LAYERS)

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        roberta_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(roberta_config)

        # Projection pour aligner les dimensions RoBERTa -> Config.EMBEDDING_DIM
        self.code_projection = nn.Linear(roberta_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention pour combiner les représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,

        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.RELU()

        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.RELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.RELU(),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Traitement des métriques avec encoder from scratch
        metrics_proj = self.metric_projection(metrics)
        metrics_proj = metrics_proj.unsqueeze(1)  # [B, 1, 384]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, 384]

        # Traitement du code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_embed = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_embed = self.code_projection(code_embed)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_embed,      # Key
            code_embed       # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# STRATÉGIE 2: Modèle entièrement pré-entraîné (encoder et decoder pré-entraînés)
class FullPretrainedModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder PRÉ-ENTRAÎNÉ pour les métriques (adaptation de RoBERTa)
        self.metrics_encoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.metrics_encoder_config.hidden_size = Config.EMBEDDING_DIM
        self.metrics_encoder = RobertaModel(self.metrics_encoder_config)

        # Projection pour les métriques numériques vers le format attendu par transformer
        self.metrics_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),

        )

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        self.code_decoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(self.code_decoder_config)

        # Projection pour aligner les dimensions
        self.code_projection = nn.Linear(self.code_decoder_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention croisée pour combiner les deux représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,

        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.RELU()

        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.RELU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.RELU(),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Projeter les métriques et les adapter au format attendu
        metrics_embedded = self.metrics_projection(metrics).unsqueeze(1)  # [B, 1, 384]

        # Créer un masque factice pour les métriques (toutes valides)
        metrics_attention_mask = torch.ones(metrics_embedded.size(0), 1).to(metrics.device)

        # Encoder les métriques avec un transformer pré-entraîné
        metrics_encoder_output = self.metrics_encoder(
            inputs_embeds=metrics_embedded,
            attention_mask=metrics_attention_mask
        )
        metrics_encoded = metrics_encoder_output.last_hidden_state  # [B, 1, 384]

        # Encoder le code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_encoded = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_encoded = self.code_projection(code_encoded)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_encoded,     # Key
            code_encoded      # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# Fonction d'équilibrage avec BorderlineSMOTE
def apply_borderline_smote(X, y, code_lines):
    """
    Applique BorderlineSMOTE pour équilibrer les classes
    """
    print(f"Distribution avant équilibrage: {Counter(y)}")

    # Vérifier qu'il y a assez d'échantillons minoritaires
    min_samples = min(Counter(y).values())
    k_neighbors = min(3, min_samples - 1) if min_samples > 1 else 1

    try:
        smote = BorderlineSMOTE(random_state=42, k_neighbors=k_neighbors, kind='borderline-1')
        X_resampled, y_resampled = smote.fit_resample(X, y)
        print("✓ BorderlineSMOTE appliqué avec succès")
    except Exception as e:
        print(f"Erreur BorderlineSMOTE: {e}, utilisation de SMOTE standard")
        smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
        X_resampled, y_resampled = smote.fit_resample(X, y)

    print(f"Distribution après équilibrage: {Counter(y_resampled)}")

    # Adapter la liste de code
    n_original = len(code_lines)
    n_resampled = len(X_resampled)

    if n_resampled > n_original:
        # Mapping pour les indices répliqués
        indices_map = []
        for i in range(n_resampled):
            if i < n_original:
                indices_map.append(i)
            else:
                # Pour les nouveaux échantillons, choisir aléatoirement
                indices_map.append(np.random.randint(0, n_original))

        code_lines_resampled = [code_lines[idx] for idx in indices_map]
    else:
        code_lines_resampled = code_lines[:n_resampled]

    return X_resampled, y_resampled, code_lines_resampled

# Calcul des poids de classe
def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    # Convertir en tensor PyTorch
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    return weight_tensor

# Prétraitement des métriques
def preprocess_metrics(df):
    """Préprocessing robuste des métriques"""
    if 'name' in df.columns:
        df = df.drop(columns=['name'])

    df['bug'] = (df['bug'] > 0).astype(int)
    features = df.drop(columns=['bug'])
    labels = df['bug'].values

    # Traiter les valeurs manquantes
    if features.isnull().any().any():
        print("Valeurs manquantes détectées, remplacement par la médiane")
        features = features.fillna(features.median())

    # Supprimer les features avec variance nulle
    var_threshold = 0.01
    low_var_cols = features.var() < var_threshold
    if low_var_cols.any():
        print(f"Suppression de {low_var_cols.sum()} colonnes avec faible variance")
        features = features.loc[:, ~low_var_cols]

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    print(f"Distribution des classes: {Counter(labels)}")

    return features_scaled, labels, scaler

# Chargement des métriques et du code
def load_code_and_metrics(metrics_path, code_path):
    """Charge les métriques et le code source"""
    df_metrics = pd.read_csv(metrics_path)
    code_lines = []

    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as f:
                            code = f.read()
                            code = code.strip()
                            if len(code) > 0:
                                code_lines.append(code)
                    except Exception as e:
                        print(f"Erreur lecture fichier {file}: {e}")
                        continue

    # S'assurer qu'on a au moins autant de fichiers code que de métriques
    if len(code_lines) < len(df_metrics):
        print(f"Attention: {len(code_lines)} fichiers code vs {len(df_metrics)} métriques")
        while len(code_lines) < len(df_metrics):
            if len(code_lines) > 0:
                code_lines.append(code_lines[len(code_lines) % len(code_lines)])
            else:
                code_lines.append("// Empty code file")

    return df_metrics, code_lines

# Préparation du dataset avec BorderlineSMOTE
def prepare_single_dataset(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset - VERSION CORRIGÉE"""
    df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
    features, labels, scaler = preprocess_metrics(df_metrics)

    # Assurer la cohérence des tailles
    min_len = min(len(features), len(code_lines), len(labels))
    features = features[:min_len]
    code_lines = code_lines[:min_len]
    labels = labels[:min_len]

    # Appliquer l'équilibrage seulement si demandé
    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, code_lines = apply_borderline_smote(features, labels, code_lines)
        except Exception as e:
            print(f"Erreur lors de l'équilibrage: {e}")
            print("Utilisation des données originales")

    # Tokenisation du code
    try:
        code_tokens = tokenizer(
            code_lines,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
    except Exception as e:
        print(f"Erreur lors de la tokenisation: {e}")
        code_tokens = {
            'input_ids': torch.zeros((len(features), 512), dtype=torch.long),
            'attention_mask': torch.zeros((len(features), 512), dtype=torch.long)
        }

    return features, code_tokens, labels, scaler

# Préparation de datasets multiples
def prepare_multiple_datasets(metrics_paths, code_paths, tokenizer, apply_balancing=True):
    """Prépare et combine plusieurs datasets - VERSION CORRIGÉE"""
    all_features, all_labels, all_code_lines = [], [], []

    # ÉTAPE 1: Charger et combiner TOUTES les données SANS équilibrage
    for metrics_path, code_path in zip(metrics_paths, code_paths):
        print(f"Chargement de {metrics_path}...")

        # Charger sans équilibrage individuel
        df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
        features, labels, scaler = preprocess_metrics(df_metrics)

        # Assurer la cohérence des tailles
        min_len = min(len(features), len(code_lines), len(labels))
        features = features[:min_len]
        code_lines = code_lines[:min_len]
        labels = labels[:min_len]

        print(f"  Dataset {metrics_path}: {Counter(labels)}")

        all_features.append(features)
        all_labels.extend(labels)
        all_code_lines.extend(code_lines)

    # ÉTAPE 2: Concaténer toutes les données
    X_combined = np.vstack(all_features)
    y_combined = np.array(all_labels)
    code_combined = all_code_lines

    print(f"\nDonnées combinées - Distribution avant équilibrage: {Counter(y_combined)}")

    # ÉTAPE 3: Appliquer BorderlineSMOTE UNE SEULE FOIS sur l'ensemble combiné
    if apply_balancing and len(set(y_combined)) > 1:
        try:
            X_balanced, y_balanced, code_balanced = apply_borderline_smote(
                X_combined, y_combined, code_combined
            )
            print(f"Données finales - Distribution après équilibrage: {Counter(y_balanced)}")
        except Exception as e:
            print(f"Erreur lors de l'équilibrage: {e}")
            print("Utilisation des données originales")
            X_balanced, y_balanced, code_balanced = X_combined, y_combined, code_combined
    else:
        X_balanced, y_balanced, code_balanced = X_combined, y_combined, code_combined

    # ÉTAPE 4: Tokeniser le code source équilibré
    try:
        print("Tokenisation du code source...")
        combined_code_tokens = tokenizer(
            code_balanced,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
        print(f"✓ Tokenisation réussie: {len(code_balanced)} échantillons")
    except Exception as e:
        print(f"Erreur lors de la tokenisation: {e}")
        combined_code_tokens = {
            'input_ids': torch.zeros((len(X_balanced), 512), dtype=torch.long),
            'attention_mask': torch.zeros((len(X_balanced), 512), dtype=torch.long)
        }

    return X_balanced, combined_code_tokens, y_balanced

# Évaluation
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# Fonction d'entraînement principale
def train_model(model_type="hybrid"):
    """Version corrigée de train_model"""
    print(f"\n{'='*50}")
    print(f"ENTRAÎNEMENT: {model_type.upper()}")
    print(f"{'='*50}")

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Définir les chemins
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/synapse/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/synapse/"

    # Versions pour l'entraînement
    train_versions = ["1.0", "1.1"]
    metrics_train_paths = [f"{base_metrics_path}synapse-{v}.csv" for v in train_versions]
    code_train_paths = [f"{base_code_path}synapse-{v}" for v in train_versions]

    # Version pour la validation
    val_version = "1.2"
    metrics_val_path = f"{base_metrics_path}synapse-{val_version}.csv"
    code_val_path = f"{base_code_path}synapse-{val_version}"

    # ✅ CORRECTION: Préparer les datasets d'entraînement avec BorderlineSMOTE appliqué UNE FOIS
    print("Préparation des données d'entraînement avec BorderlineSMOTE...")
    print("Chargement de toutes les versions d'entraînement...")
    X_train, train_code_tokens, y_train = prepare_multiple_datasets(
        metrics_train_paths, code_train_paths, tokenizer, apply_balancing=True
    )

    # Préparer le dataset de validation (sans équilibrage)
    print("Préparation des données de validation...")
    X_val, code_val_tokens, y_val, _ = prepare_single_dataset(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    # Créer les datasets
    train_ds = BugDataset(X_train, train_code_tokens, y_train)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)

    # Créer les DataLoaders
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Calculer les poids de classe
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    print(f"Poids des classes: {class_weights}")

    # Initialiser le modèle selon le type
    if model_type == "full_pretrained":
        model = FullPretrainedModel().to(Config.DEVICE)
        print("✓ Modèle avec ENCODER et DECODER pré-entraînés")
    else:
        model = HybridTransBugModel().to(Config.DEVICE)
        print("✓ Modèle avec ENCODER from scratch et DECODER pré-entraîné")

    # Optimiseur avec learning rates différenciés
    if model_type == "full_pretrained":
        # Pour le modèle entièrement pré-entraîné, utiliser des LR plus faibles pour les parties pré-entraînées
        optimizer = torch.optim.AdamW([
            {'params': model.metrics_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters()
                       if 'metrics_encoder' not in n and 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)
    else:
        # Pour le modèle hybride, seul le decoder est pré-entraîné
        optimizer = torch.optim.AdamW([
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters() if 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=True
    )

    # Loss avec pondération et label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0

    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        print(f"Epoch {epoch+1}/{Config.EPOCHS}:")
        print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
        print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")
        print(f"  Val: Sensitivité={sens:.4f}, Spécificité={spec:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            joblib.dump(cm, f"{Config.SAVE_PATH}conf_matrix_{model_type}.pkl")
            joblib.dump(val_preds, f"{Config.SAVE_PATH}val_preds_{model_type}.pkl")
            joblib.dump(val_labels, f"{Config.SAVE_PATH}val_labels_{model_type}.pkl")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 17:
            print("Early stopping déclenché")
            break

    print(f"\n{model_type.upper()} - Meilleur F1 score: {best_f1:.4f}")

    # Visualisations
    plt.figure(figsize=(15, 5))

    # Courbe 1: Loss
    plt.subplot(1, 3, 1)
    epochs_range = range(1, len(train_losses) + 1)
    plt.plot(epochs_range, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs_range, val_losses, 'r-', label='Val Loss')
    plt.title(f'{model_type.capitalize()} - Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Courbe 2: Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy')
    plt.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy')
    plt.title(f'{model_type.capitalize()} - Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Courbe 3: F1-score et G-mean
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, f1_scores, 'g-', label='F1-Score')
    plt.plot(epochs_range, gmean_scores, 'm-', label='G-Mean')
    plt.title(f'{model_type.capitalize()} - F1-Score and G-Mean')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}training_curves_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Matrice de confusion finale
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Bug', 'Bug'],
                yticklabels=['Non-Bug', 'Bug'])
    plt.title(f'{model_type.capitalize()} - Confusion Matrix (Best Model)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f"{Config.SAVE_PATH}confusion_matrix_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Sauvegarder les métriques
    results = {
        'model_type': model_type,
        'best_f1': best_f1,
        'final_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'final_sensitivity': sens,
        'final_specificity': spec,
        'final_gmean': gmean,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'f1_scores': f1_scores,
        'gmean_scores': gmean_scores
    }

    joblib.dump(results, f"{Config.SAVE_PATH}training_results_{model_type}.pkl")

    return model, results

def plot_training_results_fixed(results_hybrid, results_full, project_name):
    """Version corrigée de la génération des graphiques"""
    print(f"\n📈 GÉNÉRATION DES GRAPHIQUES POUR {project_name.upper()}")

    try:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f'Comparaison des Stratégies - {project_name.upper()}', fontsize=16, fontweight='bold')

        # Données hybrides
        hybrid_train_loss = results_hybrid['train_losses']
        hybrid_val_loss = results_hybrid['val_losses']
        hybrid_train_acc = results_hybrid['train_accuracies']
        hybrid_val_acc = results_hybrid['val_accuracies']
        hybrid_f1 = results_hybrid['f1_scores']
        hybrid_gmean = results_hybrid['gmean_scores']

        # Données full pretrained
        full_train_loss = results_full['train_losses']
        full_val_loss = results_full['val_losses']
        full_train_acc = results_full['train_accuracies']
        full_val_acc = results_full['val_accuracies']
        full_f1 = results_full['f1_scores']
        full_gmean = results_full['gmean_scores']

        # CORRECTION : Trouver la longueur minimale commune
        min_length = min(
            len(hybrid_train_loss), len(hybrid_val_loss), len(hybrid_train_acc), len(hybrid_val_acc),
            len(hybrid_f1), len(hybrid_gmean),
            len(full_train_loss), len(full_val_loss), len(full_train_acc), len(full_val_acc),
            len(full_f1), len(full_gmean)
        )

        print(f"✓ Ajustement des dimensions: {min_length} epochs communes")

        # Tronquer toutes les listes à la même longueur
        hybrid_train_loss = hybrid_train_loss[:min_length]
        hybrid_val_loss = hybrid_val_loss[:min_length]
        hybrid_train_acc = hybrid_train_acc[:min_length]
        hybrid_val_acc = hybrid_val_acc[:min_length]
        hybrid_f1 = hybrid_f1[:min_length]
        hybrid_gmean = hybrid_gmean[:min_length]

        full_train_loss = full_train_loss[:min_length]
        full_val_loss = full_val_loss[:min_length]
        full_train_acc = full_train_acc[:min_length]
        full_val_acc = full_val_acc[:min_length]
        full_f1 = full_f1[:min_length]
        full_gmean = full_gmean[:min_length]

        epochs = range(1, min_length + 1)

        # Graphique 1: Loss
        axes[0,0].plot(epochs, hybrid_train_loss, 'b-', label='Hybrid Train', linewidth=2)
        axes[0,0].plot(epochs, hybrid_val_loss, 'b--', label='Hybrid Val', linewidth=2)
        axes[0,0].plot(epochs, full_train_loss, 'r-', label='Full Pretrained Train', linewidth=2)
        axes[0,0].plot(epochs, full_val_loss, 'r--', label='Full Pretrained Val', linewidth=2)
        axes[0,0].set_title('Loss d\'Entraînement et Validation')
        axes[0,0].set_xlabel('Epochs')
        axes[0,0].set_ylabel('Loss')
        axes[0,0].legend()
        axes[0,0].grid(True, alpha=0.3)

        # Graphique 2: Accuracy
        axes[0,1].plot(epochs, hybrid_train_acc, 'b-', label='Hybrid Train', linewidth=2)
        axes[0,1].plot(epochs, hybrid_val_acc, 'b--', label='Hybrid Val', linewidth=2)
        axes[0,1].plot(epochs, full_train_acc, 'r-', label='Full Pretrained Train', linewidth=2)
        axes[0,1].plot(epochs, full_val_acc, 'r--', label='Full Pretrained Val', linewidth=2)
        axes[0,1].set_title('Accuracy d\'Entraînement et Validation')
        axes[0,1].set_xlabel('Epochs')
        axes[0,1].set_ylabel('Accuracy')
        axes[0,1].legend()
        axes[0,1].grid(True, alpha=0.3)

        # Graphique 3: F1-Score
        axes[0,2].plot(epochs, hybrid_f1, 'b-', label='Hybrid', linewidth=3)
        axes[0,2].plot(epochs, full_f1, 'r-', label='Full Pretrained', linewidth=3)
        axes[0,2].set_title('F1-Score par Epoch')
        axes[0,2].set_xlabel('Epochs')
        axes[0,2].set_ylabel('F1-Score')
        axes[0,2].legend()
        axes[0,2].grid(True, alpha=0.3)

        # Graphique 4: G-Mean
        axes[1,0].plot(epochs, hybrid_gmean, 'b-', label='Hybrid', linewidth=3)
        axes[1,0].plot(epochs, full_gmean, 'r-', label='Full Pretrained', linewidth=3)
        axes[1,0].set_title('G-Mean par Epoch')
        axes[1,0].set_xlabel('Epochs')
        axes[1,0].set_ylabel('G-Mean')
        axes[1,0].legend()
        axes[1,0].grid(True, alpha=0.3)

        # Graphique 5: Métriques finales comparées
        metrics_names = ['F1-Score', 'Accuracy', 'Sensitivity', 'Specificity', 'G-Mean']
        hybrid_metrics = [
            results_hybrid['best_f1'],
            results_hybrid['final_accuracy'],
            results_hybrid['final_sensitivity'],
            results_hybrid['final_specificity'],
            results_hybrid['final_gmean']
        ]
        full_metrics = [
            results_full['best_f1'],
            results_full['final_accuracy'],
            results_full['final_sensitivity'],
            results_full['final_specificity'],
            results_full['final_gmean']
        ]

        x_pos = np.arange(len(metrics_names))
        width = 0.35

        axes[1,1].bar(x_pos - width/2, hybrid_metrics, width, label='Hybrid', alpha=0.8, color='blue')
        axes[1,1].bar(x_pos + width/2, full_metrics, width, label='Full Pretrained', alpha=0.8, color='red')
        axes[1,1].set_title('Comparaison des Métriques Finales')
        axes[1,1].set_xlabel('Métriques')
        axes[1,1].set_ylabel('Score')
        axes[1,1].set_xticks(x_pos)
        axes[1,1].set_xticklabels(metrics_names, rotation=45)
        axes[1,1].legend()
        axes[1,1].grid(True, alpha=0.3)

        # Graphique 6: Résumé texte
        axes[1,2].axis('off')
        summary_text = (
            f"RÉSULTATS FINAUX - {project_name.upper()}\n\n"
            f"HYBRIDE:\n"
            f"• F1-Score: {results_hybrid['best_f1']:.4f}\n"
            f"• Accuracy: {results_hybrid['final_accuracy']:.4f}\n"
            f"• Sensibilité: {results_hybrid['final_sensitivity']:.4f}\n"
            f"• Spécificité: {results_hybrid['final_specificity']:.4f}\n"
            f"• G-Mean: {results_hybrid['final_gmean']:.4f}\n\n"
            f"FULL PRETRAINED:\n"
            f"• F1-Score: {results_full['best_f1']:.4f}\n"
            f"• Accuracy: {results_full['final_accuracy']:.4f}\n"
            f"• Sensibilité: {results_full['final_sensitivity']:.4f}\n"
            f"• Spécificité: {results_full['final_specificity']:.4f}\n"
            f"• G-Mean: {results_full['final_gmean']:.4f}"
        )
        axes[1,2].text(0.1, 0.9, summary_text, transform=axes[1,2].transAxes,
                       fontfamily='monospace', verticalalignment='top', fontsize=10)

        plt.tight_layout()

        # Sauvegarde
        plot_path = f"{Config.SAVE_PATH}/comparison_plot_{project_name}_FIXED.png"
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.show()

        print(f"✅ Graphiques sauvegardés: {plot_path}")

    except Exception as e:
        print(f"❌ Erreur génération graphiques pour {project_name}: {e}")
        # Fallback: générer un graphique simplifié
        self._generate_simple_plot(results_hybrid, results_full, project_name)

def _generate_simple_plot(self, results_hybrid, results_full, project_name):
    """Fallback pour générer un graphique simplifié en cas d'erreur"""
    print(f"🔄 Génération graphique simplifié pour {project_name}")

    plt.figure(figsize=(12, 8))

    # Graphique simple avec seulement les métriques disponibles
    metrics_names = ['F1-Score', 'Accuracy', 'Sensitivity', 'Specificity', 'G-Mean']
    hybrid_metrics = [
        results_hybrid['best_f1'],
        results_hybrid['final_accuracy'],
        results_hybrid['final_sensitivity'],
        results_hybrid['final_specificity'],
        results_hybrid['final_gmean']
    ]
    full_metrics = [
        results_full['best_f1'],
        results_full['final_accuracy'],
        results_full['final_sensitivity'],
        results_full['final_specificity'],
        results_full['final_gmean']
    ]

    x_pos = np.arange(len(metrics_names))
    width = 0.35

    plt.bar(x_pos - width/2, hybrid_metrics, width, label='Hybrid', alpha=0.8, color='blue')
    plt.bar(x_pos + width/2, full_metrics, width, label='Full Pretrained', alpha=0.8, color='red')
    plt.title(f'Comparaison des Métriques Finales - {project_name.upper()}')
    plt.xlabel('Métriques')
    plt.ylabel('Score')
    plt.xticks(x_pos, metrics_names, rotation=45)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    plot_path = f"{Config.SAVE_PATH}/simple_comparison_{project_name}.png"
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"✅ Graphique simplifié sauvegardé: {plot_path}")

def analyze_all_results_fixed():
    """Version corrigée de l'analyse des résultats"""
    print(f"\n{'='*80}")
    print("ANALYSE COMPLÈTE DES RÉSULTATS - VERSION CORRIGÉE")
    print(f"{'='*80}")

    # Charger les résultats globaux
    results_path = f"{Config.SAVE_PATH}/all_projects_results.pkl"

    if not os.path.exists(results_path):
        print("❌ Fichier de résultats globaux non trouvé")
        return

    all_results = joblib.load(results_path)

    # Générer les graphiques pour chaque projet
    for project, results in all_results.items():
        try:
            plot_training_results_fixed(results['hybrid'], results['full_pretrained'], project)
        except Exception as e:
            print(f"❌ Erreur génération graphiques pour {project}: {e}")
            # Essayer la version simplifiée
            _generate_simple_plot(results['hybrid'], results['full_pretrained'], project)

    # Rapport global comparatif (identique à avant)
    print(f"\n📊 RAPPORT GLOBAL COMPARATIF")
    print(f"{'='*60}")

    hybrid_wins = 0
    full_wins = 0
    ties = 0

    print(f"{'Projet':<12} {'Hybride F1':<12} {'Full F1':<12} {'Diff':<8} {'Meilleur':<10}")
    print("-" * 60)

    for project, results in all_results.items():
        hybrid_f1 = results['hybrid']['best_f1']
        full_f1 = results['full_pretrained']['best_f1']
        diff = hybrid_f1 - full_f1

        if hybrid_f1 > full_f1:
            better = "✅ Hybride"
            hybrid_wins += 1
        elif full_f1 > hybrid_f1:
            better = "✅ Full"
            full_wins += 1
        else:
            better = "⚖️ Égalité"
            ties += 1

        print(f"{project:<12} {hybrid_f1:<12.4f} {full_f1:<12.4f} {diff:+.4f}   {better:<10}")

    print("-" * 60)
    print(f"TOTAL: Hybride={hybrid_wins}, Full Pretrained={full_wins}, Égalités={ties}")

    # Recommandation finale
    if hybrid_wins > full_wins:
        print(f"\n🎯 RECOMMANDATION: La stratégie HYBRIDE est meilleure ({hybrid_wins} vs {full_wins} projets)")
    elif full_wins > hybrid_wins:
        print(f"\n🎯 RECOMMANDATION: La stratégie FULL PRETRAINED est meilleure ({full_wins} vs {hybrid_wins} projets)")
    else:
        print(f"\n🎯 RECOMMANDATION: Les deux stratégies sont équivalentes")

# REMPLACEZ simplement l'appel dans votre code existant :
if __name__ == "__main__":
    # Au lieu de analyze_all_results(), appelez :
    analyze_all_results_fixed()